# 14 — Power-law and NIE lenses

Two extensions of the SIE that capture different astrophysical regimes:

* **Power-law** lens with `kappa(x) = (3-n)/2 · x^(1-n)` (Meneghetti, *Lensing Gravitazionale*, Ch. 5.2). For ``n = 2`` this is the Singular Isothermal Sphere (SIS); for `n → 3` it approaches a point mass; for ``1 < n < 2`` the lens is more centrally concentrated than SIS.

* **Non-singular Isothermal Ellipsoid (NIE)** with a finite core radius `xi_c` (Kormann, Schneider & Bartelmann 1994, Meneghetti Ch. 5.4.2). The core regularizes the SIE central cusp and changes the **caustic topology** — for small core the standard astroid + cut survives, but for large core the radial caustic disappears and the lens stops producing radial arcs.

We map these properties numerically and compare them to the SIE baseline.

In [ ]:
# Bootstrap: make `lensing` importable when running notebooks/ directly.
import sys
from pathlib import Path
repo = Path.cwd().resolve().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import lensing as gl
device, dtype = gl.config.setup(seed=42, device="cpu")


## 1. Convergence and deflection profiles of the power law

In [ ]:
r = torch.linspace(0.01, 5.0, 200)
zeros = torch.zeros_like(r)

slopes = [1.5, 2.0, 2.5]    # 1.5 = "shallow", 2.0 = SIS, 2.5 = "steep"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for n in slopes:
    pl = gl.lens.PowerLawSpherical(theta_E=1.0, n=n)
    with torch.no_grad():
        kappa = pl.kappa(r, zeros)
        ax_def, _ = pl.deflection(r, zeros)
    axes[0].plot(r, kappa, label=f'n = {n}')
    axes[1].plot(r, ax_def, label=f'n = {n}')
for ax, ylabel, ylog in zip(axes,
                              [r'$\kappa$', r'$\alpha\;[\theta_E]$'],
                              [True, False]):
    ax.set(xlabel=r'$r/\theta_E$', ylabel=ylabel)
    if ylog: ax.set_yscale('log')
    ax.legend(); ax.grid(alpha=0.3)
axes[0].axvline(1.0, color='gray', ls=':', label=r'$\theta_E$')
axes[1].axhline(1.0, color='gray', ls=':')
plt.tight_layout(); plt.show()


**Note**: at `r = θ_E` we recover ``α = θ_E`` for **all slopes** — that's the defining property of the Einstein radius. What changes with ``n`` is the *gradient* of α, which controls magnification. SIS (``n = 2``) has constant α, so its magnification only depends on the source-plane distance from the centre.

## 2. NIE caustic topology vs. core size

Following Kormann+ 94 / Meneghetti Ch. 5.4.2, three regimes exist depending on the dimensionless ratio ``x_c / q^{3/2}``:

| Regime | Tangential caustic | Radial caustic |
|---|---|---|
| `x_c < q^{3/2}/2`           | astroid (4 cusps) | oval, contains tangential |
| `q^{3/2}/2 < x_c < q^{3/2}/(1+q)` | 2 cusps | inside tangential |
| `q^{3/2}/(1+q) < x_c < q^{1/2}/(1+q)` | survives | gone |
| `x_c > q^{1/2}/(1+q)`       | gone | gone |

We probe the topology numerically by computing det A on a fine grid and contouring the zero level.

In [ ]:
import numpy as np

def critical_curves(lens, halfwidth=2.0, n=400, dx=None):
    axis = torch.linspace(-halfwidth, halfwidth, n)
    xx, yy = torch.meshgrid(axis, axis, indexing='xy')
    if dx is None:
        dx = float(axis[1] - axis[0])
    # Finite-difference Jacobian of the lens map.
    with torch.no_grad():
        bx, by = lens.ray_trace(xx, yy)
        bxxp, _ = lens.ray_trace(xx + dx, yy)
        bxyp, _ = lens.ray_trace(xx, yy + dx)
        _, byxp = lens.ray_trace(xx + dx, yy)
        _, byyp = lens.ray_trace(xx, yy + dx)
    A11 = (bxxp - bx) / dx
    A12 = (bxyp - bx) / dx
    A21 = (byxp - by) / dx
    A22 = (byyp - by) / dx
    detA = A11 * A22 - A12 * A21
    # The corresponding caustic is the image of the critical curve.
    return axis, detA, (bx, by)

q = 0.7
cores = [0.02, 0.10, 0.30, 0.6]      # increasing core size

fig, axes = plt.subplots(2, len(cores), figsize=(4*len(cores), 8))
for j, xc in enumerate(cores):
    lens = gl.lens.NIE(theta_E=1.0, q=q, pa=0.0, core=xc)
    axis, detA, (bx, by) = critical_curves(lens, halfwidth=1.6, n=350)
    # Image-plane critical curves.
    ax = axes[0, j]
    ax.contour(axis.numpy(), axis.numpy(), detA.numpy(), levels=[0.],
               colors='C2', linewidths=1.4)
    ax.set_title(f'$x_c = {xc:.2f}$')
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    if j == 0: ax.set_ylabel('image plane $x_2$')
    # Caustic = image of the critical curve under the lens map.
    # We extract the crit curve from the contour, then ray-trace it.
    from skimage.measure import find_contours
    detA_np = detA.numpy()
    conts = find_contours(detA_np, level=0.0)
    axc = axes[1, j]
    axis_np = axis.numpy()
    for c in conts:
        # rescale row/col indices into world coordinates
        cx = np.interp(c[:, 1], np.arange(len(axis_np)), axis_np)
        cy = np.interp(c[:, 0], np.arange(len(axis_np)), axis_np)
        cxt = torch.from_numpy(cx).float()
        cyt = torch.from_numpy(cy).float()
        with torch.no_grad():
            sx, sy = lens.ray_trace(cxt, cyt)
        axc.plot(sx.numpy(), sy.numpy(), color='C3', lw=1.0)
    axc.set_aspect('equal'); axc.grid(alpha=0.3)
    if j == 0: axc.set_ylabel('source plane $y_2$')
    axc.set_xlabel('source plane $y_1$')
plt.suptitle(f'NIE critical curves (top) and caustics (bottom), q = {q}', y=1.02)
plt.tight_layout(); plt.show()


**Reading the figure**: as the core grows, the inner radial critical curve shrinks and disappears (panel 3); only the tangential caustic survives. Beyond the threshold `x_c > sqrt(q)/(1+q)` even the tangential caustic vanishes and the lens *stops producing multiple images altogether* — a useful test that very-cored systems should be filtered out from strong-lens samples.